# Preparation

In [16]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "Frechi2005" 
OLLAMA_MODEL = "qwen3-embedding:4b"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [17]:
index_name = "documents"

with driver.session() as session:
    session.run(f"DROP INDEX {index_name} IF EXISTS")
    
    session.run(f"""
        CREATE VECTOR INDEX {index_name} IF NOT EXISTS
        FOR (p:product) ON (p.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: 2560,
            `vector.similarity_function`: 'cosine'
          }}
        }}
    """)

In [20]:
ollama_embedder = OllamaEmbeddings(model=OLLAMA_MODEL)

retriever = VectorRetriever(
    driver=driver,
    index_name=index_name,
    embedder=ollama_embedder,
    return_properties=["name", "document"]
)

# Suchanfrage definieren
query = "I would like a toy for my dog!"

# Suche durchführen (Top k ähnlichste Ergebnisse)
search_results = retriever.search(query_text=query, top_k=5)

print(f"Suchanfrage: '{query}'\n")
print("Ergebnisse:")
for i, result in enumerate(search_results.items, 1):
    print(i, result)
    print(f"[{i}] Score: {result.metadata.get('score'):.4f}")

Suchanfrage: 'I would like a toy for my dog!'

Ergebnisse:
1 content='{\'name\': \'Ethical Plush Skinneeez Rabbit 23-Inch Stuffingless Dog Toy Squeaker included\', \'document\': \'- product: Ethical Plush Skinneeez Rabbit 23-Inch Stuffingless Dog Toy Squeaker included\\n- brand: PET-4-ALL\\n- description: Bring out your dog\\\'s natural hunting instinct with our realistic stuffing free skinneeez dog toys. These are some of the hottest toys available for your pet. Our stuffing free skinneeez last longer than regular plush dog toys because there is no stuffing for your dog to rip out. Now your dog can enjoy long lasting play while flip-flopping our stuffing free skinneeez. Stuffingless Dog Toy. Squeaker included.\\n- features: \\n#1: Bring out your dog\\\'s natural hunting instinct with our realistic stuffing free skinneeez dog toys.\\n#2: These are some of the hottest toys available for your pet.\\n#3: Our stuffing free skinneeez last longer than regular plush dog toys.\\n#4: Because th